## Overview

This notebook is intended to:
* Demonstrate how to programmatically use tracing functionalities available in TGI.

The five types of tracing this notebook covers are:
1. **Source**: trace upstream of a specified grid element. 
2. **All Sources**: trace all producers in the circuit of the specified grid element. Note that `Source` trace is a subset of `All Sources` trace.
3. **Down**: trace downstream of a specified grid element to all other grid elements.
4. **Connected**: trace all grid elements in the same circuit as the specified grid element. 
5. **Same Voltage**: trace all grid elements in the circuit with the same voltage as the specified grid element.

Please refer to the [main_concepts.ipynb](../2_main_concepts/main_concepts.ipynb) notebook for a high-level introduction to and simpler examples of the core views and functions available in Awesense's Energy Data Model (EDM).

---

## Set up

In [1]:
import getpass
import os
from dotenv import load_dotenv
import psycopg2
import pandas as pd
from shapely import wkb
import plotly.express as px

pd.set_option("display.max_columns", None)

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials, or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
# Checks for Google Colab: If detected,
# it bypasses reading credentials from local files and securely draws from Colab Secrets.
try:
    from google.colab import userdata
    from google.colab.userdata import SecretNotFoundError
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from contextlib import suppress
    print('☁️ Running in Google Colab. Using Colab Secrets for SQLAPI connection')
    # Look for Colab Secrets, fall back to interactive prompts if missing
    with suppress(SecretNotFoundError): os.environ['EDM_HOST'] = userdata.get('EDM_HOST')
    with suppress(SecretNotFoundError): os.environ['EDM_USER'] = userdata.get('EDM_USER')
    with suppress(SecretNotFoundError): os.environ['EDM_PASSWORD'] = userdata.get('EDM_PASSWORD')
else:
    # If running locally, loads connection parameters from a `.env` file (in this directory or parent directories)
    load_dotenv()

# Prompt the user to manually enter any missing connection parameters not found in `.env` file / Google Colab Secrets
if 'EDM_HOST' not in os.environ: os.environ['EDM_HOST'] = input('EDM server address: ').strip()
if 'EDM_USER' not in os.environ: os.environ['EDM_USER'] = input('EDM username: ')

# psycopg2/libpq automatically read the PG* environment variables, so map the shared EDM_* names onto them.
# (The password is only mapped if provided via `.env`/Colab; otherwise libpq falls back to `~/.pgpass`.)
os.environ['PGHOST'] = os.environ['EDM_HOST']
os.environ['PGUSER'] = os.environ['EDM_USER']
os.environ.setdefault('PGDATABASE', 'edm')
if 'EDM_PASSWORD' in os.environ: os.environ['PGPASSWORD'] = os.environ['EDM_PASSWORD']

print('Verifying SQLAPI connection parameters/credentials with connection attempt')

try:
    # Test database connection
    conn = psycopg2.connect('')
    conn.close()
    print('✅ SQLAPI connection parameters/credentials verified')
except psycopg2.OperationalError:
    print('⚠️ SQLAPI credentials check failed (assume missing/incorrect password, but double-check `.env` / secrets!)')
    # Prompt user securely for password
    os.environ['PGPASSWORD'] = getpass.getpass('Enter EDM Password manually: ')

# Keeps `%sql` result tables rendering correctly on newer versions of prettytable
import prettytable
if 'DEFAULT' not in vars(prettytable): prettytable.DEFAULT = prettytable.TableStyle.DEFAULT

# Load the SQL extension and connect
get_ipython().run_line_magic('reload_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://')

Verifying SQLAPI connection parameters/credentials with connection attempt


✅ SQLAPI connection parameters/credentials verified


**Custom Functions**

In [3]:
def query_to_df(result):
    """
    Transform sql query result to dataframe, and
    expand out `meta` column into separate columns.
    """

    # Turn a query output into a Python dataframe.
    df = result.DataFrame()

    # Separate the `meta` column currently saved as JSONB into individual columns.
    # Replace NaN values to empty string for ease of spotting non-NaN values.
    df = pd.concat(
        [df.drop(["meta"], axis=1), df["meta"].apply(pd.Series).fillna("")], axis=1
    )

    # Transform the hex string form of geometry to coordinates.
    df["geometry"] = df["geometry"].apply(lambda x: wkb.loads(x, hex=True))

    return df


def plot_pie(df, interest_colname, title):
    """
    Compute counts of each value in `interest_colname` column,
    plot a pie chart of the breakdown, and
    return the counts dataframe.
    """

    # Create a dataframe of counts of each value.
    df_cts = (
        df[interest_colname]
        .value_counts()
        .to_frame(name="count")
        .rename_axis(interest_colname)
        .reset_index()
    )

    # Create a pie plot.
    fig = px.pie(df_cts, names=interest_colname, values="count", title=title)

    # Show the count values, percentages and labels for the pie plot.
    fig.update_traces(textposition="inside", textinfo="percent+label+value")
    fig.show()

    # Return the count dataframe sorted alphabetically by in the input column.
    return df_cts.sort_values(interest_colname)


def phase_table(df):
    """
    Return a table of phase breakdowns by grid element type.
    """

    # Create a dataframe of counts by both types and phases.
    df_counts = df[["type", "phases"]].value_counts().to_frame().reset_index()

    # Rename the first column as 'count'.
    df_counts = df_counts.rename(columns={0: "count"})

    # Pivot the dataframe to have phases as columns.
    df_phases = df_counts.pivot_table(index="type", columns="phases", fill_value=0)

    # Return the count dataframe sorted alphabetically by `type`.
    return df_phases.sort_values("type")

---

## Example

### Input Parameters

In [4]:
# PARAMETERS (Cell Tag: parameters)
# Do not split this cell structure. Papermill injects automated
# runtime overrides immediately below this block.
# If you don't want to be asked for input for these parameters, you can set them here.
grid_id = None  # Target grid identifier (e.g., 'awefice')
element_type = None  # Element type (e.g., 'Transformer')
grid_element_id = None  # Transformer for tracing (e.g., 'transformer_6')
trace_option = None  # Tracing option (e.g., 'Down')
meter_id = None  # Meter ID for tracing parent transformer (e.g., 'm_7')

We'll take a look at the `awefice` grid. In the database, this property is referred to as a `grid_id`.

In [5]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
grid_id = grid_id or input("Enter grid ID: ").strip()  # awefice

Check when this grid was last updated. 

In [6]:
%%sql

SELECT last_updated
FROM grid
WHERE grid_id = :grid_id;

 * postgresql://


1 rows affected.


last_updated
2021-01-01 20:00:00+00:00


The below are different element types available in the `awefice` grid.

In [7]:
%%sql

SELECT DISTINCT type as element_type
FROM grid_element
WHERE grid_id = :grid_id
ORDER BY element_type;

 * postgresql://


15 rows affected.


element_type
ACLineSegment
Battery
Busbar
Cabinet
CircuitBreaker
Disconnector
EVCharger
Fuse
Jumper
Meter


Enter one of the element types from the above output to explore further in the `grid_element` view.

In [8]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
element_type = (
    element_type or input("Enter the element type of interest: ").strip()
)  # Transformer

The below are all available grid element ID for the specified grid `type`.

In [9]:
%%sql

SELECT grid_element_id
FROM grid_element
WHERE grid_id = :grid_id
    AND type = :element_type
ORDER BY grid_element_id;

 * postgresql://


10 rows affected.


grid_element_id
transformer_16
transformer_2
transformer_26
transformer_31
transformer_36
transformer_43
transformer_47
transformer_6
transformer_63
transformer_92


Choose one of the grid element ID from the above output to conduct tracing on.

In [10]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
grid_element_id = (
    grid_element_id or input("Enter the grid element id of interest: ").strip()
)  # transformer_6

### Trace

Choose one of the tracing options: `Source`, `All Sources`, `Down`, `Connected`, and `Same Voltage`.

In [11]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
trace_option = trace_option or input("Enter the tracing option: ").strip()  # Down

Call an appropriate SQL query based on the specified tracing option.

In [12]:
if trace_option.lower() == 'source':

    # Set the third argument to `true` to retrieve the top feeder.
    result = %sql SELECT * \
            FROM grid_get_sources(:grid_id, :grid_element_id, 'true');

elif trace_option.lower() == 'all sources':

    # Set the third argument to `false` to retrieve all feeders.
    result = %sql SELECT * \
            FROM grid_get_sources(:grid_id, :grid_element_id, 'false');

elif trace_option.lower() == 'down':

    # Set the third argument to `false` to exclude itself from the output.
    result = %sql SELECT * \
            FROM grid_get_downstream(:grid_id, :grid_element_id, 'false');

elif trace_option.lower() == 'connected':

    result = %sql SELECT * \
            FROM grid_get_connected(:grid_id, :grid_element_id);

elif trace_option.lower() == 'same voltage':

    result = %sql SELECT * \
            FROM grid_get_same_voltage(:grid_id, :grid_element_id);

else:
    print('Invalid input for tracing option. Please enter one of the following: Source, All Sources, Down, Connected.')

 * postgresql://


97 rows affected.


In [13]:
# Turn the sql output to a Python dataframe.
df = query_to_df(result)

# Return the first few rows.
df.head()

,grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,name,label,sdp_id,latitude,meter_id,longitude,lv_feeder,mv_feeder,ownership,make_model,enclosure_id,voltage_level,capacity_usable,commission_date,capacity_nominal,manufacture_date,efficiency_round_trip,max_peak_charge_power,parent_transformer_id,max_peak_discharge_power,max_continuous_charge_power,max_continuous_discharge_power,feeder_name,is_operation_status,rating_kva,max_current,max_voltage,is_dc_charger,charging_level,connector_type,active_charging_power,active_generation_power,rating_a,system,materials,latitude_1,latitude_2,line_length,longitude_1,longitude_2,is_insulated,ph_ph_voltage,wire_diameter,hvmv_parent_element,mvlv_parent_element,is_ground_deployable,address,interval,estimated,read_type,tariff_id,meter_type,is_critical,meter_number,zone_level_1,zone_level_2,zone_level_3,zone_level_4,energy_source,maximal_demand,meter_multiplier,source_rating_kw,type_of_consumer,connected_to_lvmv,replacement_cycle,service_drop_type,standard_profile_id,maximal_demand_units,business_identifier_id,ac_size,dc_size,racking,panel_area,tilt_angle,azimuth_angle,dc_to_ac_ratio,number_of_panels,panel_make_model,panel_rated_power,degradation_annual,generation_capacity,inverter_make_model,degradation_25_years,control_method,conducting_status,normal_operation_status,switchable_element_type,load_loss,no_load_loss,primary_voltage,maintenance_type,installation_type,secondary_voltage,winding_configuration
0,awefice,batt_19,Battery,battery,ABC,None,True,True,False,None,con_222_t2,VCN.batt_19,both,NaN,POINT (-123.105327544793 49.2679075484494),batt_19,batt_19,,,m_19,,,,private,Panasonic 11.4 kWh EverVolt,,LV,11.4,,13.5,,84.44,5.5,transformer_63,5.5,4.8,4.8,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,awefice,batt_24,Battery,battery,ABC,None,True,True,False,None,con_312_t2,VCN.batt_24,both,NaN,POINT (-123.101676761526 49.2682843622986),batt_24,batt_24,,,m_24,,,,private,Nissan XStorage,,LV,9.58,,10.08,,95.04,6.0,transformer_92,6.0,3.6,3.6,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,awefice,busbar_65,Busbar,busbar,ABC,None,False,False,False,None,bb_con_65,VCN.busbar_65,1to2,NaN,LINESTRING (-123.104680618614 49.2683153194707...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,awefice,busbar_94,Busbar,busbar,ABC,None,False,False,False,None,bb_con_94,VCN.busbar_94,1to2,NaN,LINESTRING (-123.102717395475 49.2682834411654...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,awefice,evch_20,EVCharger,ev_charger,ABC,None,True,True,False,None,con_228_t2,VCN.evch_20,both,NaN,POINT (-123.104161611466 49.2683270934028),,evch_20,,,m_20,,,,private,Grizzl-E EV Charger [GR1-14-24-P],,LV,,,,,,,transformer_63,,,,,,,16.0,240.0,False,level_2,NEMA 14-30,3840.0,0.0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


The above dataframe can be processed further as needed. For example, TGI only returns a top feeder for the `Source` option whereas the notebook here returns all grid_elements. Applying the `is_producer==True` filter on the `df` will mirror the same functionality as in TGI.

In [14]:
# Filter the dataframe to rows with is_producer=True.
df[df["is_producer"] == True]

,grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,name,label,sdp_id,latitude,meter_id,longitude,lv_feeder,mv_feeder,ownership,make_model,enclosure_id,voltage_level,capacity_usable,commission_date,capacity_nominal,manufacture_date,efficiency_round_trip,max_peak_charge_power,parent_transformer_id,max_peak_discharge_power,max_continuous_charge_power,max_continuous_discharge_power,feeder_name,is_operation_status,rating_kva,max_current,max_voltage,is_dc_charger,charging_level,connector_type,active_charging_power,active_generation_power,rating_a,system,materials,latitude_1,latitude_2,line_length,longitude_1,longitude_2,is_insulated,ph_ph_voltage,wire_diameter,hvmv_parent_element,mvlv_parent_element,is_ground_deployable,address,interval,estimated,read_type,tariff_id,meter_type,is_critical,meter_number,zone_level_1,zone_level_2,zone_level_3,zone_level_4,energy_source,maximal_demand,meter_multiplier,source_rating_kw,type_of_consumer,connected_to_lvmv,replacement_cycle,service_drop_type,standard_profile_id,maximal_demand_units,business_identifier_id,ac_size,dc_size,racking,panel_area,tilt_angle,azimuth_angle,dc_to_ac_ratio,number_of_panels,panel_make_model,panel_rated_power,degradation_annual,generation_capacity,inverter_make_model,degradation_25_years,control_method,conducting_status,normal_operation_status,switchable_element_type,load_loss,no_load_loss,primary_voltage,maintenance_type,installation_type,secondary_voltage,winding_configuration
0,awefice,batt_19,Battery,battery,ABC,None,True,True,False,None,con_222_t2,VCN.batt_19,both,NaN,POINT (-123.105327544793 49.2679075484494),batt_19,batt_19,,,m_19,,,,private,Panasonic 11.4 kWh EverVolt,,LV,11.4,,13.5,,84.44,5.5,transformer_63,5.5,4.8,4.8,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,awefice,batt_24,Battery,battery,ABC,None,True,True,False,None,con_312_t2,VCN.batt_24,both,NaN,POINT (-123.101676761526 49.2682843622986),batt_24,batt_24,,,m_24,,,,private,Nissan XStorage,,LV,9.58,,10.08,,95.04,6.0,transformer_92,6.0,3.6,3.6,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,awefice,evch_20,EVCharger,ev_charger,ABC,None,True,True,False,None,con_228_t2,VCN.evch_20,both,NaN,POINT (-123.104161611466 49.2683270934028),,evch_20,,,m_20,,,,private,Grizzl-E EV Charger [GR1-14-24-P],,LV,,,,,,,transformer_63,,,,,,,16.0,240.0,False,level_2,NEMA 14-30,3840.0,0.0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
5,awefice,evch_24,EVCharger,ev_charger,ABC,None,True,True,False,None,con_312_t2,VCN.evch_24,both,NaN,POINT (-123.101676761526 49.2682843622986),,evch_24,,,m_24,,,,private,ChargePoint CPE 200 [CPE200T-S-CHD-CMB],,LV,,,,,,,transformer_92,,,,,,,125.0,500.0,True,level_3,"[CHAdeMO,CCS1]",50000.0,0.0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
76,awefice,pv_19,Photovoltaic,photovoltaic,ABC,None,True,True,False,None,con_222_t2,VCN.pv_19,both,NaN,POINT (-123.105327544793 49.2679075484494),pv_19,pv_19,,,m_19,,,,private,,,LV,,2014-07-26,,,,,transformer_63,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,1813.0,2075.0,carport,1.915056,39.0,174.0,1.144,5.0,SunPower Maxeon3 [SPR-MAX6-415-BLK-E3-AC],415.0,0.25,1813.0,Fronius Primo,6.25,,,,,,,,,,,
77,awefice,pv_24,Photovoltaic,photovoltaic,ABC,None,True,True,False,None,con_312_t2,VCN.pv_24,both,NaN,POINT (-123.101676761526 49.2682843622986),pv_24,pv_24,,,m_24,,,,private,,,LV,,2018-03-20,,,,,transformer_92,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,49984.0,62530.0,ground,1.748536,40.0,172.0,1.251,169.0,Panasonic EverVolt [EVPV370],370.0,0.25,49984.0,SMA\tSunny Boy,6.25,,,,,,,,,,,
78,awefice,pv_27,Photovoltaic,photovoltaic,A,None,True,True,False,None,con_287_t2,VCN.pv_27,both,NaN,POINT (-123.103364337434 49.2678756811234),pv_27,pv_27,,,m_27,,,,private,,,LV,,2012-08-26,,,,,transformer_92,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,

For the demonstration of this notebook, we will continue using the full dataframe with unfiltered data.

In [15]:
# Create a pie plot for the breakdown of grid element types.
df_types = plot_pie(
    df, "type", "Breakdown of Grid Element Types for " + trace_option.title() + " Trace"
)

In [16]:
# Return a table of grid element type counts.
df_types

,type,count
0,ACLineSegment,52
6,Battery,2
7,Busbar,2
4,CircuitBreaker,5
9,Disconnector,2
8,EVCharger,2
2,Fuse,8
3,Jumper,6
1,Meter,12
5,Photovoltaic,3


In [17]:
# Create a pie plot for the breakdown of phases.
df_phases = plot_pie(
    df, "phases", "Breakdown of Phases for " + trace_option.title() + " Trace"
)

In [18]:
# Return a table of phase counts.
df_phases

,phases,count
1,A,3
0,ABC,91
2,B,2
3,C,1


In [19]:
# Create a table of phase counts by grid element types.
phase_table(df)

count                
phases             A   ABC    B    C
type                                
ACLineSegment    0.0  52.0  0.0  0.0
Battery          0.0   2.0  0.0  0.0
Busbar           0.0   2.0  0.0  0.0
CircuitBreaker   0.0   5.0  0.0  0.0
Disconnector     0.0   2.0  0.0  0.0
EVCharger        0.0   2.0  0.0  0.0
Fuse             0.0   8.0  0.0  0.0
Jumper           0.0   6.0  0.0  0.0
Meter            2.0   7.0  2.0  1.0
Photovoltaic     1.0   2.0  0.0  0.0
Switch           0.0   1.0  0.0  0.0
Transformer      0.0   2.0  0.0  0.0

### Combining Trace Functions

Combining multiple trace functions provides the flexibility to find an answer to additional questions.

For example, a grid element's parent transformer can be identified by: 
1. Returning a list of transformers in the upstream using `grid_get_sources()`.
2. Entering the list of transformers from step 1 to `grid_get_same_voltage()` to get all grid elements with the same voltage as a given transformer.
3. Matching the input grid element to the returned grid elements from step 2 and filter to the respective transformer as the final output.

In [20]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
meter_id = meter_id or input("Enter the meter ID of interest: ").strip()  # m_7

In [21]:
%%sql

SELECT ggs.grid_element_id AS transformer
FROM grid_get_sources(:grid_id, :meter_id, true) ggs
LEFT JOIN grid_get_same_voltage(:grid_id, ggs.grid_element_id) ggsv
    ON true
WHERE ggs.type = 'Transformer'
    AND ggsv.grid_element_id = :meter_id;


 * postgresql://


1 rows affected.


transformer
transformer_36


---